In [1]:
# Step 1: Set Up Your Environment
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
# Step 1: Set Up Your Environment
from pyspark.sql import SparkSession

In [3]:
# Initialize Spark session
spark = SparkSession.builder.appName("house").getOrCreate()

In [4]:
# Step 2: Load the Data
data = spark.read.csv("Housing Price data set.csv", header=True, inferSchema=True)
data.show(3)

+---+-------+-------+--------+-------+-------+--------+-------+--------+-----+-----+--------+--------+
|_c0|  price|lotsize|bedrooms|bathrms|stories|driveway|recroom|fullbase|gashw|airco|garagepl|prefarea|
+---+-------+-------+--------+-------+-------+--------+-------+--------+-----+-----+--------+--------+
|  1|42000.0|   5850|       3|      1|      2|     yes|     no|     yes|   no|   no|       1|      no|
|  2|38500.0|   4000|       2|      1|      1|     yes|     no|      no|   no|   no|       0|      no|
|  3|49500.0|   3060|       3|      1|      1|     yes|     no|      no|   no|   no|       0|      no|
+---+-------+-------+--------+-------+-------+--------+-------+--------+-----+-----+--------+--------+
only showing top 3 rows



In [5]:
# Step 3: Index Categorical Columns
categorical_columns = ["driveway", "recroom", "fullbase", "gashw", "airco", "prefarea"]
indexers = [StringIndexer(inputCol=col, outputCol=f"{col}_index", handleInvalid="keep") for col in categorical_columns]

In [6]:
# Step 4: Assemble Feature Columns into a Single Vector
feature_columns = ["lotsize", "bedrooms", "bathrms", "stories", "garagepl"] + [f"{col}_index" for col in categorical_columns]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

In [7]:
# Step 5: Apply Feature Scaling
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)

In [8]:
# Step 6: Create the Pipeline
pipeline = Pipeline(stages=indexers + [assembler, scaler])

# Step 7: Apply Transformations
data_preprocessed = pipeline.fit(data).transform(data)

TypeError: Cannot recognize a pipeline stage of type <class 'list'>.

In [ ]:
data_preprocessed.show(2)

In [ ]:
# Step 8: Split the Data into Training and Testing Sets
train_data, test_data = data_preprocessed.randomSplit([0.8, 0.2], seed=1)

In [ ]:
# Step 9: Build and Train the Regression Model
lr = LinearRegression(featuresCol="scaled_features", labelCol="price")  # 'price' is the target variable
lr_model = lr.fit(train_data)

In [ ]:
# Step 10: Make Predictions
predictions = lr_model.transform(test_data)

In [ ]:
predictions.show(2)

In [ ]:
# Step 11: Evaluate the Model
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Test RMSE = {rmse}")

In [ ]:
# Step 12: Save the Model
lr_model.save("/path/to/save/model")